# DMALA alpha scan at several training checkpoints

This notebook loads a trained one-layer MLP BEBM, selects 5 checkpoints approximately linearly spaced between the beginning and the end of training, and for each checkpoint measures:

- the exponential decorrelation time $\tau_{\mathrm{exp}}(\alpha)$,
- the mean DMALA acceptance rate as a function of $\alpha$.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from rbms.io import load_params
from rbms.utils import get_saved_updates

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

MODEL_FILE = "pcd_trains/BEBM_MLP_MNIST_h500_DMALA100_lr1e-2_100k.h5"
NUM_CHECKPOINTS = 5
ALPHAS = np.array([0.05, 0.08, 0.1, 0.13, 0.15, 0.18, 0.2, 0.25, 0.3, 0.4, 0.5, 0.8, 1.0], dtype=float)
NUM_CHAINS = 256
NUM_STEPS = 20000
MAX_LAG = 10000
BETA = 1.0
FIT_MIN_RHO = 0.05
FIT_MAX_RHO = 0.6
SEED = 0

saved_updates = get_saved_updates(MODEL_FILE)
target_updates = np.linspace(int(saved_updates[0]), int(saved_updates[-1]), NUM_CHECKPOINTS)
scan_updates = []
for target in target_updates:
    index = np.argmin(np.abs(saved_updates - target))
    scan_updates.append(int(saved_updates[index]))
scan_updates = sorted(set(scan_updates))

print("device:", device)
print("model file:", MODEL_FILE)
print("selected checkpoints:", scan_updates)
print("alphas:", ALPHAS)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (2168886850.py, line 15)

In [ ]:
def dmala_step_energy(energy, visible, alpha, beta=1.0):
    visible_in = visible.detach().requires_grad_(True)
    current_energy = energy(visible_in).view(-1)
    grad = torch.autograd.grad(current_energy.sum(), visible_in)[0]

    forward_logits = -0.5 * beta * grad + (2.0 * visible_in - 1.0) / (2.0 * alpha)
    forward_prob = torch.sigmoid(forward_logits)
    proposal = torch.bernoulli(forward_prob).detach()

    proposal_in = proposal.requires_grad_(True)
    proposal_energy = energy(proposal_in).view(-1)
    proposal_grad = torch.autograd.grad(proposal_energy.sum(), proposal_in)[0]

    reverse_logits = -0.5 * beta * proposal_grad + (2.0 * proposal_in - 1.0) / (2.0 * alpha)

    with torch.no_grad():
        log_q_forward = -F.binary_cross_entropy_with_logits(
            forward_logits.detach(), proposal, reduction="none"
        ).sum(dim=1)
        log_q_reverse = -F.binary_cross_entropy_with_logits(
            reverse_logits.detach(), visible, reduction="none"
        ).sum(dim=1)

        log_acceptance = (
            -beta * proposal_energy.detach()
            + beta * current_energy.detach()
            + log_q_reverse
            - log_q_forward
        )

        accept = torch.log(torch.rand_like(log_acceptance)) < log_acceptance
        new_visible = torch.where(accept[:, None], proposal, visible)

    return new_visible, accept.float().mean().item()


def spin_autocorr_fft(spins, max_lag, component_batch=4096, fft_device=None):
    if fft_device is None:
        fft_device = spins.device

    spins = spins.to(torch.float32)
    num_steps_plus_one, num_chains, num_visibles = spins.shape
    max_lag = min(int(max_lag), num_steps_plus_one - 1)

    mean_spin = spins.mean(dim=(0, 1))
    denominator = (1.0 - mean_spin.pow(2)).sum().clamp_min(1e-12)

    centered = spins - mean_spin.view(1, 1, -1)
    n_time = centered.shape[0]
    n_fft = 1 << ((2 * n_time - 1).bit_length())

    autocorr_sum = torch.zeros(max_lag + 1, device=fft_device)

    for start in range(0, num_visibles, component_batch):
        stop = min(start + component_batch, num_visibles)
        block = centered[:, :, start:stop].to(fft_device)
        block = block.permute(1, 2, 0).reshape(-1, n_time)

        fft_block = torch.fft.rfft(block, n=n_fft, dim=1)
        power = fft_block * fft_block.conj()
        corr = torch.fft.irfft(power, n=n_fft, dim=1)[:, : max_lag + 1]

        counts = torch.arange(n_time, n_time - max_lag - 1, -1, device=fft_device, dtype=torch.float32)
        corr = corr / counts

        autocorr_sum += corr.sum(dim=0)

    rho = autocorr_sum / denominator.to(fft_device)
    rho[0] = 1.0
    return rho.cpu().numpy()


def collect_spin_trajectory_dmala(energy, num_visibles, num_chains, num_steps, alpha, beta=1.0, seed=0):
    torch.manual_seed(seed)
    visible = torch.bernoulli(
        torch.full((num_chains, num_visibles), 0.5, device=device, dtype=dtype)
    )

    spin_trajectory = []
    acceptances = []

    for step in range(num_steps + 1):
        spin_trajectory.append((2.0 * visible.detach().cpu() - 1.0).to(torch.float32))

        if step < num_steps:
            visible, acceptance = dmala_step_energy(energy, visible, alpha=alpha, beta=beta)
            acceptances.append(acceptance)

    spins = torch.stack(spin_trajectory, dim=0)
    mean_acceptance = float(np.mean(acceptances)) if acceptances else np.nan
    return spins, mean_acceptance


def fit_tau_exp(rho, fit_min_rho=0.05, fit_max_rho=0.6):
    lags = np.arange(len(rho))
    fit_mask = (lags > 0) & np.isfinite(rho) & (rho > fit_min_rho) & (rho < fit_max_rho)

    if fit_mask.sum() < 2:
        return np.nan

    slope, intercept = np.polyfit(lags[fit_mask], np.log(rho[fit_mask]), deg=1)
    return -1.0 / slope if slope < 0 else np.inf


In [ ]:
results = {}

for update in scan_updates:
    params = load_params(MODEL_FILE, update, device=device, dtype=dtype)
    energy = params.energy
    num_visibles = params.num_visibles

    results[update] = {}
    for alpha in tqdm(ALPHAS, desc=f"alpha scan update {update}"):
        spins, acceptance = collect_spin_trajectory_dmala(
            energy=energy,
            num_visibles=num_visibles,
            num_chains=NUM_CHAINS,
            num_steps=NUM_STEPS,
            alpha=float(alpha),
            beta=BETA,
            seed=SEED,
        )

        rho = spin_autocorr_fft(
            spins=spins,
            max_lag=MAX_LAG,
            fft_device=device,
        )

        tau_exp = fit_tau_exp(
            rho=rho,
            fit_min_rho=FIT_MIN_RHO,
            fit_max_rho=FIT_MAX_RHO,
        )

        results[update][float(alpha)] = {
            "tau_exp": tau_exp,
            "acceptance": acceptance,
        }

        print(f"update={update}, alpha={alpha:.3f}: acceptance={acceptance:.4f}, tau_exp={tau_exp:.2f}")


alpha scan update 1:   0%|          | 0/9 [00:39<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

for update in scan_updates:
    alphas = np.array(sorted(results[update].keys()))
    tau_values = np.array([results[update][alpha]["tau_exp"] for alpha in alphas])
    ax.plot(alphas, tau_values, marker="o", label=f"update {update}")

ax.set_title(r"Exponential decorrelation time $\tau_{\mathrm{exp}}$ vs $\alpha$")
ax.set_xlabel(r"$\alpha$")
ax.set_ylabel(r"$\tau_{\mathrm{exp}}$")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

for update in scan_updates:
    alphas = np.array(sorted(results[update].keys()))
    acceptance_values = np.array([results[update][alpha]["acceptance"] for alpha in alphas])
    ax.plot(alphas, acceptance_values, marker="o", label=f"update {update}")

ax.set_title(r"DMALA acceptance rate vs $\alpha$")
ax.set_xlabel(r"$\alpha$")
ax.set_ylabel("acceptance rate")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
